In [ ]:
### What we know before engineering features

Before writing a single feature, we have already identified several decisions 
that will directly affect the code:

| Variable | Issue | Action |
|---|---|---|
| RHD018 | In months, not years | Divide by 12 |
| RHQ010 | Code 0 = not yet started | Handle explicitly, not as NaN |
| RHQ160 | Code 11 = 11 or more | Top-coded; treat as 11 for continuous use |
| RHD167 | Code 5 = 5 or more | Top-coded; treat as 5 for continuous use |
| RHQ162 | Code 3 = borderline GDM | Treat as Yes (conservative) |
| RHQ070 | Ordinal ranges, not exact ages | Use midpoint imputation for Age_Menopause |
| RHQ074/076/078 | Infertility block, not surgical history | Used for Has_Infertility feature |
| RHQ163/166/169 | Not present in this release | HDP and excess GWG cannot be derived |
| RHQ542A–D | Multi-select, sparse (<4%) | Build composite HRT_Type, exclude from clustering |

Variables retained for descriptive use only (< 10% coverage):
RHQ070, RHQ197, RHQ200, RHQ332, RHQ542B, RHQ542C, RHQ542D, RHD018

In [ ]:
### What we know before engineering features

Before writing a single feature, we have already identified several decisions 
that will directly affect the code:

| Variable | Issue | Action |
|---|---|---|
| RHD018 | In months, not years | Divide by 12 |
| RHQ010 | Code 0 = not yet started | Handle explicitly, not as NaN |
| RHQ160 | Code 11 = 11 or more | Top-coded; treat as 11 for continuous use |
| RHD167 | Code 5 = 5 or more | Top-coded; treat as 5 for continuous use |
| RHQ162 | Code 3 = borderline GDM | Treat as Yes (conservative) |
| RHQ070 | Ordinal ranges, not exact ages | Use midpoint imputation for Age_Menopause |
| RHQ074/076/078 | Infertility block, not surgical history | Used for Has_Infertility feature |
| RHQ163/166/169 | Not present in this release | HDP and excess GWG cannot be derived |
| RHQ542A–D | Multi-select, sparse (<4%) | Build composite HRT_Type, exclude from clustering |

Variables retained for descriptive use only (< 10% coverage):
RHQ070, RHQ197, RHQ200, RHQ332, RHQ542B, RHQ542C, RHQ542D, RHD018

In [ ]:
---
## Section 4 · Feature Engineering

We now build features domain by domain. Each subsection explains the clinical 
reasoning before the code. The domains and their order:

1. **Pregnancy history** — gravidity, parity, parity category
2. **Adverse pregnancy outcomes** — GDM, macrosomia, APO score
3. **Reproductive span** — menarche age, menopause age, span calculation
4. **Menopausal status** — derived from surgical history, periods, and menopause age
5. **Surgical history** — hysterectomy, bilateral oophorectomy
6. **Infertility markers** — Has_Infertility, PID history
7. **Hormone therapy** — ever used, type composite

The engineered features are collected into a single `feat` dataframe throughout, 
then assembled and exported at the end of Section 5.

In [ ]:
### 4.1 · Pregnancy History

**Clinical context:** Accurately characterizing pregnancy history requires distinguishing 
between two related but distinct obstetric concepts that are frequently conflated in 
non-clinical data science work:

- **Gravidity** (`Gravidity` ← RHQ160): total number of times pregnant, including 
  miscarriages, stillbirths, ectopic pregnancies, and terminations. This is the 
  denominator of reproductive experience.
- **Parity** (`Parity` ← RHD167): total number of deliveries, regardless of outcome. 
  Twins delivered in one event count as one delivery. This is the obstetric definition 
  used in CVD risk literature.

For CKM phenotyping, parity is the more relevant exposure — prospective studies 
associate it with cardiovascular risk in a dose-response pattern, with both nulliparity 
and grand multiparity (≥5) carrying elevated risk through different mechanisms.

**Variables used:**

| Variable | Feature | Notes |
|---|---|---|
| RHQ131 | Ever_Pregnant | Gate variable — drives structural missingness downstream |
| RHD143 | Currently_Pregnant | Exclusion flag for certain analyses; ages 20–44 only |
| RHQ160 | Gravidity | Top-coded at 11; retained as 11 |
| RHD167 | Parity | Top-coded at 5; retained as 5 |
| RHQ171 | Live_Births | Used to derive Pregnancy_Loss |
| RHD180 | Age_First_Birth | Bottom-coded at 17, top-coded at 45 |
| RHD190 | Age_Last_Birth | Bottom-coded at 17, top-coded at 45 |

**A note on top-coding:** NHANES caps certain count variables at a maximum value 
to protect participant privacy and simplify data collection. For gravidity, exact 
counts are recorded up to 10 pregnancies — anyone with 11 or more is simply 
recorded as 11. For parity, exact counts go up to 4 deliveries, with 5 or more 
collapsed into 5. These are not missing values — they are real responses, just 
imprecise at the high end. We keep them as-is rather than converting to NaN, 
accepting a small loss of precision for a clinically rare group.

**Derived features:**
- `Pregnancy_Loss` = Gravidity − Live_Births. Captures miscarriage, stillbirth, 
  ectopic, and termination burden. Recurrent pregnancy loss is associated with 
  antiphospholipid syndrome and thrombotic risk — relevant CKM context.
- `Childbearing_Span` = Age_Last_Birth − Age_First_Birth. Only calculated for 
  women with more than one delivery.
- `Parity_Category`: ordinal grouping aligned with CVD risk literature 
  (Nulliparous / Primiparous / Multiparous 2–3 / Multiparous 4–5 / Grand Multiparous).

In [ ]:
# Initialize our engineered feature dataframe
feat = pd.DataFrame(index=df_clean.index)

# ── Ever Pregnant ──────────────────────────────────────────────────────────
feat['Ever_Pregnant'] = recode_yes_no(df_clean['RHQ131'])

# ── Currently Pregnant ─────────────────────────────────────────────────────
# RHD143: only asked of women aged 20–44; NaN for everyone else is structural
feat['Currently_Pregnant'] = recode_yes_no(df_clean['RHD143'])

# ── Gravidity (total pregnancies) ──────────────────────────────────────────
# RHQ160: total times pregnant including miscarriages, stillbirths, abortions
# Top-coded at 11 (11 or more) — we retain 11 as the numeric value
# RHQ141 does NOT exist in this release — RHQ160 is the correct column
grav_raw = clean_nhanes_col(df_clean['RHQ160']).copy()

high_grav = grav_raw[grav_raw > 15]
if len(high_grav) > 0:
    print(f'⚠️  Implausible gravidity values (>15): {len(high_grav)} participants')
    print(high_grav.value_counts())

feat['Gravidity'] = grav_raw
# Women who said never pregnant get Gravidity = 0
feat.loc[feat['Ever_Pregnant'] == 0, 'Gravidity'] = 0

# ── Parity (total deliveries) ──────────────────────────────────────────────
# RHD167: counts deliveries (not children — twins = 1 delivery)
# Top-coded at 5 (5 or more) — we retain 5 as the numeric value
# This is the obstetric definition of parity, distinct from gravidity above
par_raw = clean_nhanes_col(df_clean['RHD167']).copy()
feat['Parity'] = par_raw
feat.loc[feat['Ever_Pregnant'] == 0, 'Parity'] = 0

# ── Live Births ────────────────────────────────────────────────────────────
# RHQ171: subset of deliveries that resulted in live birth
# Useful for distinguishing pregnancy loss burden from live birth parity
feat['Live_Births'] = clean_nhanes_col(df_clean['RHQ171']).copy()
feat.loc[feat['Ever_Pregnant'] == 0, 'Live_Births'] = 0

# ── Pregnancy Loss ─────────────────────────────────────────────────────────
# Derived: pregnancies that did not result in a live birth delivery
# Captures miscarriages, stillbirths, terminations, ectopic pregnancies
feat['Pregnancy_Loss'] = feat['Gravidity'] - feat['Live_Births']
# Negative values would indicate a data inconsistency — flag and null
impossible = feat['Pregnancy_Loss'] < 0
if impossible.sum() > 0:
    print(f'⚠️  {impossible.sum()} participants with Pregnancy_Loss < 0 — setting to NaN')
    feat.loc[impossible, 'Pregnancy_Loss'] = np.nan

# ── Parity Category ────────────────────────────────────────────────────────
# Based on total deliveries (RHD167), not gravidity
# Categories align with obstetric literature on CVD risk stratification
def categorize_parity(p):
    if pd.isna(p):   return np.nan
    elif p == 0:     return 'Nulliparous'
    elif p == 1:     return 'Primiparous'
    elif p <= 3:     return 'Multiparous_2_3'
    elif p <= 5:     return 'Multiparous_4_5'
    else:            return 'Grand_Multiparous'

feat['Parity_Category'] = feat['Parity'].apply(categorize_parity)

# ── Birth Timing ───────────────────────────────────────────────────────────
# RHD180/RHD190: bottom-coded at 17 (17 or younger), top-coded at 45+
# We retain the sentinel values as-is — they are real ages, just imprecise
feat['Age_First_Birth'] = clean_nhanes_col(df_clean['RHD180'])
feat['Age_Last_Birth']  = clean_nhanes_col(df_clean['RHD190'])

# Childbearing span — only meaningful if more than one birth
feat['Childbearing_Span'] = feat['Age_Last_Birth'] - feat['Age_First_Birth']
feat.loc[feat['Parity'] <= 1, 'Childbearing_Span'] = np.nan

# ── Summary ────────────────────────────────────────────────────────────────
print('=== Pregnancy History Features ===')
print(f"Ever_Pregnant:     {feat['Ever_Pregnant'].value_counts().to_dict()}")
print(f"Currently_Pregnant:{feat['Currently_Pregnant'].value_counts().to_dict()}")
print(f"\nGravidity:\n{feat['Gravidity'].describe().round(1)}")
print(f"\nParity:\n{feat['Parity'].describe().round(1)}")
print(f"\nParity_Category:\n{feat['Parity_Category'].value_counts(dropna=False)}")
print(f"\nPregnancy_Loss:\n{feat['Pregnancy_Loss'].describe().round(1)}")

In [ ]:
### Validating Pregnancy History Features

Before moving forward we examine the output of the pregnancy history cell to confirm 
the features behave as expected both numerically and clinically.

**What looks right:**

- **Gravidity vs Parity difference:** Mean gravidity is 2.7 and mean parity is 2.1 — 
  a difference of ~0.6 pregnancies per woman. This gap represents average pregnancy 
  loss burden (miscarriages, stillbirths, ectopic pregnancies, terminations) and is 
  consistent with population-level data where approximately 10–20% of recognized 
  pregnancies end in loss.
- **Pregnancy_Loss maximum of 10:** Plausible — a woman recorded with 11 pregnancies 
  (top-coded) and 1 delivery would produce exactly this value. No cause for concern.
- **Nulliparous count of 640:** Matches the `Ever_Pregnant = 0` count exactly, 
  confirming that our backfill logic (assigning Parity = 0 to never-pregnant women) 
  worked correctly and there are no internal inconsistencies between these two features.
- **Parity maximum of 5:** Confirms top-coding is being handled correctly — no values 
  above 5 appear despite some women having more than 5 deliveries.

**What required a closer look:**

The `Parity_Category` column showed 1,269 NaN values — larger than the 640 nulliparous 
women alone would explain. This meant approximately 629 ever-pregnant women had no 
parity category assigned, pointing to missingness in `RHD167` (total deliveries) rather 
than a coding error.

Investigation revealed only 18 ever-pregnant women had missing `RHD167` — far fewer 
than the 629 we expected. The remaining gap is accounted for by women whose 
`Ever_Pregnant` status was itself missing (NaN), meaning they were never asked the 
downstream delivery questions. This is structural missingness driven by skip logic, 
not a data quality problem.

The 18 women with confirmed missing parity were then investigated further — see the 
cell below for the full reasoning and imputation decision.

In [ ]:
# How many participants are teenagers (12-19) who were never asked RHQ131?
# We need P_DEMO for age — first cross-module linkage in this notebook

demo = pd.read_sas('../data/raw/P_DEMO.XPT', format='xport', encoding='utf-8')
demo = demo[['SEQN', 'RIDAGEYR']].set_index('SEQN')

# Merge age into our working dataframe temporarily
age_check = df_clean.join(demo['RIDAGEYR'])

teenagers = age_check[age_check['RIDAGEYR'] < 20]
print(f'Participants aged 12–19: {len(teenagers)}')
print(f'Of these, RHQ131 is NaN: {teenagers["RHQ131"].isna().sum()}')
print(f'Of these, RHQ131 has a value: {teenagers["RHQ131"].notna().sum()}')

adults = age_check[age_check['RIDAGEYR'] >= 20]
print(f'\nParticipants aged 20+: {len(adults)}')
print(f'Of these, RHQ131 is NaN: {adults["RHQ131"].isna().sum()}')
print(f'Of these, RHQ131 has a value: {adults["RHQ131"].notna().sum()}')

In [ ]:
# Investigate NaN Parity_Category among ever-pregnant women
nan_parity = feat[(feat['Ever_Pregnant'] == 1) & (feat['Parity_Category'].isna())]
print(f'Ever-pregnant women with missing Parity_Category: {len(nan_parity)}')
print(f'Of these, Gravidity is known for: {nan_parity["Gravidity"].notna().sum()}')
print(f'Of these, Parity (RHD167) is missing for: {nan_parity["Parity"].isna().sum()}')

# 18 / 3,412 ever-pregnant women (0.5%) — acceptable level of missingness
# These women answered RHQ131 (ever pregnant) but did not complete RHD167 (deliveries)
# Likely due to interview interruption or skip logic edge cases
# No imputation applied — they will carry NaN for all parity-dependent features
print(f'\n✓ {len(nan_parity)} women ({len(nan_parity)/feat["Ever_Pregnant"].sum()*100:.1f}%) '
      f'with missing parity')

**Is the missingness random or systematic?**

We should check whether those 18 women are similar to the rest of the ever-pregnant group in terms of age, gravidity, and other characteristics. If they're systematically different — say, all older, or all with high gravidity — then even 18 women could introduce bias.

In [ ]:
print('Gravidity among the 18 missing parity women:')
print(nan_parity['Gravidity'].value_counts().sort_index())

print('\nCompare to overall ever-pregnant gravidity distribution:')
print(feat[feat['Ever_Pregnant']==1]['Gravidity'].describe().round(1))

In [ ]:
# Visualize parity distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of parity count
parity_counts = feat['Parity'].dropna()
axes[0].hist(parity_counts, bins=range(0, int(parity_counts.max()) + 2),
             color='steelblue', edgecolor='white', align='left')
axes[0].set_xlabel('Number of Pregnancies')
axes[0].set_ylabel('Count')
axes[0].set_title('Parity Distribution')

# Parity category breakdown
cat_order = ['Nulliparous', 'Primiparous', 'Multiparous_2_3', 'Multiparous_4_5', 'Grand_Multiparous']
cat_counts = feat['Parity_Category'].value_counts()[cat_order]
axes[1].barh(cat_counts.index, cat_counts.values, color='steelblue')
axes[1].set_xlabel('Count')
axes[1].set_title('Parity Category Breakdown')

plt.tight_layout()
plt.savefig('../data/processed/fig_parity.png', bbox_inches='tight')
plt.show()
print('Saved: fig_parity.png')

In [ ]:
---
### 4.2 · Adverse Pregnancy Outcomes (APO)

**Clinical context:** The 2021 AHA/ACC chest pain guidelines and subsequent CKM framework specifically recognize adverse pregnancy outcomes as **female-specific cardiovascular risk enhancers**. The three most actionable for our phenotyping are:

1. **Gestational Diabetes Mellitus (GDM):** ~7× increased lifetime T2D risk, 2× CVD risk independent of subsequent diabetes.
2. **Macrosomia (birth weight ≥9 lbs / ~4,000g):** Marker of fetal hyperinsulinism and maternal insulin resistance during pregnancy, even without formal GDM diagnosis.
3. **Hypertensive disorders of pregnancy (HDP):** Pre-eclampsia is associated with 2-4× increased future hypertension and 2× CVD risk.

We will construct an **APO Score** — a simple additive count of adverse outcomes — that can serve as a continuous exposure variable in downstream clustering.

In [ ]:
# ── Gestational Diabetes ────────────────────────────────────────────────────
# RHQ162: 1=Yes, 2=No, 3=Borderline → we treat 3 as Yes (conservative)
if 'RHQ162' in df_clean.columns:
    gdm_raw = df_clean['RHQ162'].copy()

    print('RHQ162 raw value counts (after missing code removal):')
    print(gdm_raw.value_counts(dropna=False))
    print('\nNote: Code 3 = Borderline GDM → will be coded as GDM = Yes (conservative approach)')

    # 1 or 3 → Yes (1.0); 2 → No (0.0); NaN → NaN
    feat['GDM'] = gdm_raw.map({1.0: 1.0, 2.0: 0.0, 3.0: 1.0})

    # Readable label for exploration
    feat['GDM_Label'] = gdm_raw.map({1.0: 'Yes', 2.0: 'No', 3.0: 'Yes (borderline)'})

    n_gdm = feat['GDM'].sum()
    n_asked = feat['GDM'].notna().sum()
    print(f'\nGDM prevalence among those asked: {n_gdm/n_asked*100:.1f}% (n={int(n_gdm)})')

In [ ]:
# ── Macrosomia ─────────────────────────────────────────────────────────────
# RHQ172: Baby weighing ≥9 lbs at birth
if 'RHQ172' in df_clean.columns:
    feat['Macrosomia'] = recode_yes_no(df_clean['RHQ172'])

    n_mac = feat['Macrosomia'].sum()
    n_asked = feat['Macrosomia'].notna().sum()
    print(f'Macrosomia prevalence among those asked: {n_mac/n_asked*100:.1f}% (n={int(n_mac)})')

# ── Hypertensive Disorders of Pregnancy ────────────────────────────────────
# RHQ163 = Hypertension during pregnancy
# RHQ166 = Toxemia/pre-eclampsia
# We create a combined HDP flag (Yes if either is Yes)
hdp_cols = [c for c in ['RHQ163', 'RHQ166'] if c in df_clean.columns]
if hdp_cols:
    hdp_binary = pd.DataFrame({
        col: recode_yes_no(df_clean[col]) for col in hdp_cols
    })
    # Any HDP: 1 if at least one component is 1; 0 if all are 0; NaN if all are NaN
    feat['HDP'] = hdp_binary.max(axis=1)  # max of (0/1/NaN) effectively = OR logic

    n_hdp = feat['HDP'].sum()
    n_asked = feat['HDP'].notna().sum()
    print(f'HDP prevalence among those asked:        {n_hdp/n_asked*100:.1f}% (n={int(n_hdp)})')

# ── Excess Gestational Weight Gain ─────────────────────────────────────────
if 'RHQ169' in df_clean.columns:
    feat['Excess_GWG'] = recode_yes_no(df_clean['RHQ169'])

In [ ]:
# ── APO Score ──────────────────────────────────────────────────────────────
# Additive count of adverse pregnancy outcomes.
# Range: 0 (no APOs) to 3+ (multiple APOs).
# Only calculated for women who were ever pregnant.

apo_components = [c for c in ['GDM', 'Macrosomia', 'HDP'] if c in feat.columns]

# Sum components; NaN treated as 0 for the total (conservative approach).
# We explicitly set APO_Score to NaN for never-pregnant women.
feat['APO_Score'] = feat[apo_components].fillna(0).sum(axis=1)

# NaN out women who were never pregnant (APO score is not meaningful for them)
if 'Ever_Pregnant' in feat.columns:
    feat.loc[feat['Ever_Pregnant'] == 0, 'APO_Score'] = np.nan

# Also NaN out women for whom we have no APO data at all
no_apo_data = feat[apo_components].isna().all(axis=1)
feat.loc[no_apo_data, 'APO_Score'] = np.nan

print('=== APO Score Distribution ===')
print(feat['APO_Score'].value_counts(dropna=False).sort_index())
print(f"\nMean APO Score (ever-pregnant): {feat.loc[feat['Ever_Pregnant']==1, 'APO_Score'].mean():.2f}")
print(f"Any APO (score ≥ 1): {(feat['APO_Score'] >= 1).sum():,} women")

In [ ]:
# Visualize APO patterns
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# APO score distribution
score_counts = feat['APO_Score'].dropna().value_counts().sort_index()
axes[0].bar(score_counts.index.astype(int), score_counts.values, color='coral')
axes[0].set_xlabel('APO Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Adverse Pregnancy Outcome Score')
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(1))

# Individual APO prevalence
apo_prev = {
    col: feat.loc[feat[col].notna(), col].mean() * 100
    for col in apo_components
    if col in feat.columns
}
axes[1].barh(list(apo_prev.keys()), list(apo_prev.values()), color='coral')
axes[1].set_xlabel('Prevalence (%)')
axes[1].set_title('APO Component Prevalence (among those asked)')

plt.tight_layout()
plt.savefig('../data/processed/fig_apo.png', bbox_inches='tight')
plt.show()

In [ ]:
---
### 4.3 · Reproductive Span

**Clinical context:** The reproductive span — the interval between menarche and menopause — is a proxy for cumulative estrogen exposure. Both extremes carry risk:
- **Early menarche (≤11):** Associated with obesity, insulin resistance, and earlier CVD onset
- **Late menopause (≥52):** Associated with higher breast cancer and endometrial cancer risk, but potentially protective for CVD
- **Short reproductive span (<30 years):** May indicate early ovarian aging or surgical menopause, associated with elevated CVD risk

For the reproductive span calculation, we need **both** ages — and we'll have missing data on menopause for pre-menopausal women (which is not a data problem; it's expected).

In [ ]:
# ── Age at Menarche ────────────────────────────────────────────────────────
# RHQ010: Reported age; RHD018: Estimated age (if exact not known)
if 'RHQ010' in df_clean.columns:
    menarche = df_clean['RHQ010'].copy()

    # Supplement with estimated age where exact is missing
    if 'RHD018' in df_clean.columns:
        menarche = menarche.fillna(df_clean['RHD018'])

    # Sanity range: menarche typically 8–18 years
    invalid_menarche = menarche[(menarche < 8) | (menarche > 20)]
    if len(invalid_menarche) > 0:
        print(f'Implausible menarche ages (outside 8–20): {len(invalid_menarche)}')
        print(invalid_menarche.value_counts())
        menarche[(menarche < 8) | (menarche > 20)] = np.nan

    feat['Age_Menarche'] = menarche

    print('Age at Menarche:')
    print(feat['Age_Menarche'].describe().round(1))

# ── Early Menarche Flag ────────────────────────────────────────────────────
feat['Early_Menarche'] = (feat['Age_Menarche'] <= 11).astype(float)
feat.loc[feat['Age_Menarche'].isna(), 'Early_Menarche'] = np.nan
print(f"\nEarly menarche (≤11): {feat['Early_Menarche'].sum():.0f} women")

In [ ]:
# ── Age at Menopause ────────────────────────────────────────────────────────
# RHQ060: Age at which periods stopped (natural menopause)
# RHQ197: Age at last menstrual period (complementary)
if 'RHQ060' in df_clean.columns:
    menopause_age = df_clean['RHQ060'].copy()

    # Supplement with last menstrual period age if available
    if 'RHQ197' in df_clean.columns:
        menopause_age = menopause_age.fillna(df_clean['RHQ197'])

    # Sanity range: menopause typically 35–65 years
    invalid_meno = menopause_age[(menopause_age < 30) | (menopause_age > 70)]
    if len(invalid_meno) > 0:
        print(f'Implausible menopause ages (outside 30–70): {len(invalid_meno)}')
        menopause_age[(menopause_age < 30) | (menopause_age > 70)] = np.nan

    feat['Age_Menopause'] = menopause_age

    print('Age at Menopause (among post-menopausal):')
    print(feat['Age_Menopause'].describe().round(1))
    print(f'  Premature menopause (<40): {(feat["Age_Menopause"] < 40).sum():.0f}')
    print(f'  Early menopause (40–44):   {((feat["Age_Menopause"] >= 40) & (feat["Age_Menopause"] < 45)).sum():.0f}')

# ── Reproductive Span ──────────────────────────────────────────────────────
if 'Age_Menarche' in feat.columns and 'Age_Menopause' in feat.columns:
    feat['Reproductive_Span'] = feat['Age_Menopause'] - feat['Age_Menarche']

    # Negative spans are impossible — flag and null
    impossible_span = feat['Reproductive_Span'] < 0
    if impossible_span.sum() > 0:
        print(f'\nImpossible negative spans set to NaN: {impossible_span.sum()}')
        feat.loc[impossible_span, 'Reproductive_Span'] = np.nan

    print(f"\nReproductive Span (post-menopausal women only):")
    print(feat['Reproductive_Span'].describe().round(1))

In [ ]:
---
### 4.4 · Menopausal Status

**Clinical context:** Menopausal status is a critical CKM covariate — the loss of estrogen's cardioprotective effects after menopause accelerates CVD risk accumulation in women. But "menopausal status" in cross-sectional data is nuanced:

We derive it using a **decision tree logic** that integrates:
1. Current period regularity (RHQ031)
2. Surgical history (hysterectomy, oophorectomy)
3. Age at menopause (if reported)

**Classification hierarchy:**
```
Bilateral oophorectomy → Surgical Menopause
Hysterectomy (without both ovaries removed) → Post-Hysterectomy (indeterminate)
Regular periods currently → Pre-Menopausal
No regular periods + menopause age reported → Post-Menopausal (Natural)
No regular periods + no surgery + no menopause age → Peri-Menopausal / Unknown
```

In [ ]:
# ── Surgical History ────────────────────────────────────────────────────────
if 'RHQ074' in df_clean.columns:
    feat['Hysterectomy'] = recode_yes_no(df_clean['RHQ074'])

if 'RHQ078' in df_clean.columns:
    feat['Bilateral_Oophorectomy'] = recode_yes_no(df_clean['RHQ078'])

# Ages at surgery (continuous)
if 'RHQ076' in df_clean.columns:
    feat['Age_Hysterectomy'] = df_clean['RHQ076']

if 'RHQ080' in df_clean.columns:
    feat['Age_Oophorectomy'] = df_clean['RHQ080']

# ── Regular Periods ─────────────────────────────────────────────────────────
if 'RHQ031' in df_clean.columns:
    feat['Regular_Periods'] = recode_yes_no(df_clean['RHQ031'])

print('Surgical history summary:')
if 'Hysterectomy' in feat.columns:
    print(f"  Hysterectomy:          {feat['Hysterectomy'].value_counts().to_dict()}")
if 'Bilateral_Oophorectomy' in feat.columns:
    print(f"  Bilateral Oophorectomy:{feat['Bilateral_Oophorectomy'].value_counts().to_dict()}")
if 'Regular_Periods' in feat.columns:
    print(f"  Regular Periods:       {feat['Regular_Periods'].value_counts().to_dict()}")

In [ ]:
# ── Derive Menopausal Status ────────────────────────────────────────────────
def derive_menopausal_status(row):
    """
    Derive menopausal status from surgical history, period regularity,
    and menopause age.

    Returns one of:
      'Pre_Menopausal'          - currently has regular periods, no hysterectomy
      'Post_Menopausal_Natural'  - periods stopped, no surgery, menopause age known
      'Post_Menopausal_Surgical' - bilateral oophorectomy
      'Post_Hysterectomy'        - hysterectomy without bilateral oophorectomy
      'Peri_Menopausal'          - no regular periods, no surgery, menopause age unknown
      NaN                        - insufficient data
    """
    bso        = row.get('Bilateral_Oophorectomy', np.nan)
    hyst       = row.get('Hysterectomy', np.nan)
    reg_period = row.get('Regular_Periods', np.nan)
    meno_age   = row.get('Age_Menopause', np.nan)

    # Priority 1: Surgical menopause
    if bso == 1:
        return 'Post_Menopausal_Surgical'

    # Priority 2: Hysterectomy without oophorectomy
    if hyst == 1 and bso != 1:
        return 'Post_Hysterectomy'

    # Priority 3: Currently regular periods → pre-menopausal
    if reg_period == 1:
        return 'Pre_Menopausal'

    # Priority 4: No regular periods + menopause age known → post-menopausal
    if reg_period == 0 and pd.notna(meno_age):
        return 'Post_Menopausal_Natural'

    # Priority 5: No regular periods but no menopause age → peri or unknown
    if reg_period == 0:
        return 'Peri_Menopausal'

    return np.nan


status_cols = ['Bilateral_Oophorectomy', 'Hysterectomy', 'Regular_Periods', 'Age_Menopause']
available_status_cols = [c for c in status_cols if c in feat.columns]

feat['Menopausal_Status'] = feat[available_status_cols].apply(
    derive_menopausal_status, axis=1
)

print('=== Menopausal Status ===')
print(feat['Menopausal_Status'].value_counts(dropna=False))

In [ ]:
# Create binary Post_Menopausal flag for modeling
# (combines natural + surgical post-menopausal; excludes post-hysterectomy
#  as ovarian status is indeterminate)
feat['Post_Menopausal'] = feat['Menopausal_Status'].isin(
    ['Post_Menopausal_Natural', 'Post_Menopausal_Surgical']
).astype(float)
feat.loc[feat['Menopausal_Status'].isna(), 'Post_Menopausal'] = np.nan

print(f"Post_Menopausal (binary): {feat['Post_Menopausal'].value_counts().to_dict()}")

---
### 4.5 · Infertility / PCOS Proxy

**Clinical context:** Polycystic ovary syndrome (PCOS) is the most common endocrine disorder in reproductive-age women (~10% prevalence) and is a recognized CKM risk factor — associated with insulin resistance, dyslipidemia, and hypertension independent of BMI. NHANES doesn't have a direct PCOS diagnosis code, but `RHQ540` ("Ever told by a doctor you had difficulty conceiving or were infertile") serves as a reasonable proxy for PCOS-associated reproductive dysfunction.

This is a **proxy, not a diagnosis** — we will document this limitation clearly.

In [ ]:
if 'RHQ540' in df_clean.columns:
    feat['Has_Infertility'] = recode_yes_no(df_clean['RHQ540'])

    n_inf = feat['Has_Infertility'].sum()
    n_asked = feat['Has_Infertility'].notna().sum()
    print(f'Self-reported infertility/difficulty conceiving:')
    print(f'  N = {int(n_inf)} of {int(n_asked)} asked ({n_inf/n_asked*100:.1f}%)')
    print(f'  [Note: This is a PCOS proxy, not a PCOS diagnosis]')

# ── Hormone Therapy ─────────────────────────────────────────────────────────
if 'RHQ554' in df_clean.columns:
    feat['Hormone_Therapy'] = recode_yes_no(df_clean['RHQ554'])
    print(f"\nHormone therapy use: {feat['Hormone_Therapy'].value_counts().to_dict()}")